In [147]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

In [148]:
df = pd.read_csv('data/credit_card_cleaned.csv')
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V26,V27,V28,Amount,Class,SmallTransaction,LargeTransaction,RegularTransaction,Hour,Day
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.189115,0.133558,-0.021053,149.62,0,False,False,True,0.0,0.0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,0.125895,-0.008983,0.014724,2.69,0,True,False,False,0.0,0.0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,-0.139097,-0.055353,-0.059752,378.66,0,False,True,False,0.0,0.0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.221929,0.062723,0.061458,123.50,0,False,False,True,0.0,0.0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,0.502292,0.219422,0.215153,69.99,0,False,False,True,0.0,0.0


### Data modelling using logistic regression
1. Seperate y and X sets for modelling
2. Scale the time columns('Time', 'Hour' and 'Day')
3. Get the confusion matrix (and classification report) to see how well our model predicted

In [149]:
#Seperating class label and attributes
X = df.drop(columns=['Class'])
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2, stratify=y)

In [150]:
#Scalling the various time attributes and amount attributes due 
#to a large difference between anonymized values and other attributes
attributes = ['Time', 'Amount', 'Hour', 'Day']
scaler = StandardScaler()
X_train[attributes] = scaler.fit_transform(X_train[attributes])
X_test[attributes] = scaler.transform(X_test[attributes])

In [151]:
#Training the model, class_weight must be passed with the argument 'balanced'
model_logistic = LogisticRegression(class_weight='balanced', max_iter=10000)
model_logistic.fit(X_train, y_train)
y_pred = model_logistic.predict(X_test)

In [152]:
print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

--- Confusion Matrix ---
[[55173  1691]
 [    4    94]]


In [153]:
print("--- Classification Report ---")
print(classification_report(y_test, y_pred))

--- Classification Report ---
              precision    recall  f1-score   support

           0       1.00      0.97      0.98     56864
           1       0.05      0.96      0.10        98

    accuracy                           0.97     56962
   macro avg       0.53      0.96      0.54     56962
weighted avg       1.00      0.97      0.98     56962



### Data Modelling using Ensemble methods
1. Create a random forest model and compare the classification report to the logistic regression report

In [163]:
model_forest = RandomForestClassifier(n_estimators=100,
                                      max_depth=12,
                                      class_weight='balanced',
                                      random_state=42)
model_forest.fit(X_train, y_train)
y_pred_forest = model_forest.predict(X_test)


In [164]:
print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred_forest))

--- Confusion Matrix ---
[[56844    20]
 [   13    85]]


In [166]:
print("--- Classification Report ---")
print(classification_report(y_test, y_pred_forest))

--- Classification Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.81      0.87      0.84        98

    accuracy                           1.00     56962
   macro avg       0.90      0.93      0.92     56962
weighted avg       1.00      1.00      1.00     56962



### Improving the Random Forest Model
As seen by the classification report between the two models, the RF performs significantly better. What we need now is to 
improve the models ability to actually predict fraud cases (which was slightly worse than logistice regression)

In [170]:
#We want to see what probabilities each instance is getting from our RF model (note: P(Class0) + P(Class1) = 1 for each row)
rf_y_proba = model_forest.predict_proba(X_test)[:, 1]

#By default, predict() has a thresholf of 0.5
threshold = 0.4
rf_y_proba_custom = (rf_y_proba >= threshold).astype(int)


In [171]:
print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, rf_y_proba_custom))

--- Confusion Matrix ---
[[56841    23]
 [   12    86]]


In [172]:
print("--- Classification Report ---")
print(classification_report(y_test, rf_y_proba_custom))

--- Classification Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.79      0.88      0.83        98

    accuracy                           1.00     56962
   macro avg       0.89      0.94      0.92     56962
weighted avg       1.00      1.00      1.00     56962

